In [ ]:
## part 3 Python Reconciliation Workflow

In [16]:
import pandas as pd

In [18]:
ledger_df = pd.read_csv('/content/ledger.csv')
gateway_df = pd.read_csv('/content/gateway.csv')

print("Ledger DataFrame head:")
display(ledger_df.head())

print("\nGateway DataFrame head:")
display(gateway_df.head())

Ledger DataFrame head:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,850.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R004,2026-03-02,M003,2100.0,success,Card
4,R005,2026-03-03,M004,7200.0,success,Card



Gateway DataFrame head:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,900.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R005,2026-03-03,M004,7200.0,failed,Card
4,R006,2026-03-03,M002,950.0,success,UPI


In [19]:
# Check for duplicate rows
duplicate_rows = df[df.duplicated()]
print("Number of duplicate rows:", len(duplicate_rows))
if not duplicate_rows.empty:
    print("\nDuplicate rows (first 5 if any):")
    display(duplicate_rows.head())

Number of duplicate rows: 0


In [20]:
# Check for null values in each column
print("\nNumber of null values per column:")
null_counts = df.isnull().sum()
display(null_counts)

if null_counts.any():
    print("\nColumns with null values:")
    display(null_counts[null_counts > 0])
else:
    print("\nNo null values found in any column.")


Number of null values per column:


,0
batch_id,0
merchant_id,0
merchant_name,0
region,0
settlement_id,0
amount_usd,0
status,0
processed_at,0
bank_name,0
bank_country,0



No null values found in any column.


### Identify Records Missing in Gateway

We will now identify transactions that are present in the `ledger_df` but do not have a corresponding record in the `gateway_df`. This is typically done by performing a left merge and then checking for `NaN` values in columns that should have been populated from the right (gateway) DataFrame.

In [21]:
# Perform a left merge to find records in ledger_df not in gateway_df
# Assuming 'transaction_id' is a common unique identifier
merged_df = ledger_df.merge(gateway_df, on='transaction_id', how='left', suffixes=('_ledger', '_gateway'))

# Identify rows where gateway-specific columns are null, indicating they are missing in gateway_df
missing_in_gateway = merged_df[merged_df['status_gateway'].isnull()]

print("Records from Ledger missing in Gateway:")
display(missing_in_gateway[['transaction_id', 'amount_usd_ledger', 'status_ledger']])

Records from Ledger missing in Gateway:


,transaction_id,amount_usd_ledger,status_ledger
3,R004,2100.0,success
9,R010,2500.0,success


### Identify Records Missing in Ledger

We will now identify transactions that are present in the `gateway_df` but do not have a corresponding record in the `ledger_df`. This is typically done by performing a right merge and then checking for `NaN` values in columns that should have been populated from the left (ledger) DataFrame.

In [22]:
# Perform a right merge to find records in gateway_df not in ledger_df
# Assuming 'transaction_id' is a common unique identifier
merged_df = ledger_df.merge(gateway_df, on='transaction_id', how='right', suffixes=('_ledger', '_gateway'))

# Identify rows where ledger-specific columns are null, indicating they are missing in ledger_df
missing_in_ledger = merged_df[merged_df['status_ledger'].isnull()]

print("Records from Gateway missing in Ledger:")
display(missing_in_ledger[['transaction_id', 'amount_usd_gateway', 'status_gateway']])

Records from Gateway missing in Ledger:


,transaction_id,amount_usd_gateway,status_gateway
8,R011,1800.0,success


### Identify Amount Mismatches

Now, we will compare the `amount_usd` for transactions that are present in both the `ledger_df` and the `gateway_df` to identify any discrepancies. This is done by performing an inner merge and then filtering for rows where the amounts differ.

In [23]:
# Perform an inner merge to get only transactions present in both DataFrames
matched_transactions = ledger_df.merge(gateway_df, on='transaction_id', how='inner', suffixes=('_ledger', '_gateway'))

# Identify rows where the amount_usd does not match
amount_mismatches = matched_transactions[matched_transactions['amount_usd_ledger'] != matched_transactions['amount_usd_gateway']]

print("Records with Amount Mismatches:")
display(amount_mismatches[['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway', 'status_ledger', 'status_gateway']])

Records with Amount Mismatches:


,transaction_id,amount_usd_ledger,amount_usd_gateway,status_ledger,status_gateway
1,R002,850.0,900.0,success,success
6,R008,640.0,600.0,success,success


### Identify Status Mismatches

Finally, we will compare the `status` for transactions that are present in both the `ledger_df` and the `gateway_df` to identify any discrepancies. This is done by performing an inner merge and then filtering for rows where the statuses differ.

In [24]:
# Use the already matched_transactions DataFrame from the previous step
# Identify rows where the status does not match
status_mismatches = matched_transactions[matched_transactions['status_ledger'] != matched_transactions['status_gateway']]

print("Records with Status Mismatches:")
display(status_mismatches[['transaction_id', 'status_ledger', 'status_gateway', 'amount_usd_ledger', 'amount_usd_gateway']])

Records with Status Mismatches:


,transaction_id,status_ledger,status_gateway,amount_usd_ledger,amount_usd_gateway
3,R005,success,failed,7200.0,7200.0


### Final Reconciliation Report

Now, we will combine all the identified discrepancies into a single reconciliation report. Each type of discrepancy will be labeled with an `issue_type` for easy identification.

In [25]:
# Prepare missing_in_gateway records
report_missing_in_gateway = missing_in_gateway[['transaction_id', 'amount_usd_ledger', 'status_ledger']]
report_missing_in_gateway = report_missing_in_gateway.rename(columns={'amount_usd_ledger': 'amount_usd', 'status_ledger': 'status'})
report_missing_in_gateway['issue_type'] = 'missing_in_gateway'

# Prepare missing_in_ledger records
report_missing_in_ledger = missing_in_ledger[['transaction_id', 'amount_usd_gateway', 'status_gateway']]
report_missing_in_ledger = report_missing_in_ledger.rename(columns={'amount_usd_gateway': 'amount_usd', 'status_gateway': 'status'})
report_missing_in_ledger['issue_type'] = 'missing_in_ledger'

# Prepare amount_mismatches records
report_amount_mismatches = amount_mismatches[['transaction_id', 'amount_usd_ledger', 'amount_usd_gateway', 'status_ledger', 'status_gateway']]
report_amount_mismatches['issue_type'] = 'amount_mismatch'

# Prepare status_mismatches records
report_status_mismatches = status_mismatches[['transaction_id', 'status_ledger', 'status_gateway', 'amount_usd_ledger', 'amount_usd_gateway']]
report_status_mismatches['issue_type'] = 'status_mismatch'

# Combine all reports into a single DataFrame
final_reconciliation_report = pd.concat([
    report_missing_in_gateway,
    report_missing_in_ledger,
    report_amount_mismatches,
    report_status_mismatches
], ignore_index=True)

print("Final Reconciliation Report:")
display(final_reconciliation_report)

Final Reconciliation Report:


/tmp/ipykernel_8962/3170837164.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  report_amount_mismatches['issue_type'] = 'amount_mismatch'
/tmp/ipykernel_8962/3170837164.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  report_status_mismatches['issue_type'] = 'status_mismatch'


,transaction_id,amount_usd,status,issue_type,amount_usd_ledger,amount_usd_gateway,status_ledger,status_gateway
0,R004,2100.0,success,missing_in_gateway,NaN,NaN,NaN,NaN
1,R010,2500.0,success,missing_in_gateway,NaN,NaN,NaN,NaN
2,R011,1800.0,success,missing_in_ledger,NaN,NaN,NaN,NaN
3,R002,NaN,NaN,amount_mismatch,850.0,900.0,success,success
4,R008,NaN,NaN,amount_mismatch,640.0,600.0,success,success
5,R005,NaN,NaN,status_mismatch,7200.0,7200.0,success,failed


### Summary of Discrepancies

This section provides a high-level overview of the types and counts of discrepancies identified during the reconciliation process.

In [26]:
summary_metrics = final_reconciliation_report['issue_type'].value_counts().reset_index()
summary_metrics.columns = ['Issue Type', 'Count']

print("Summary of Reconciliation Discrepancies:")
display(summary_metrics)

Summary of Reconciliation Discrepancies:


,Issue Type,Count
0,missing_in_gateway,2
1,amount_mismatch,2
2,missing_in_ledger,1
3,status_mismatch,1


In [ ]:
## part 4 JSON Normalisation

In [1]:
import pandas as pd
import json

In [2]:
json_file_path = '/content/api_response_sample.json'

with open(json_file_path, 'r') as f:
    data = json.load(f)

# Display the loaded JSON data
print(json.dumps(data, indent=2))

{
  "generated_at": "2026-03-07T10:00:00Z",
  "source": "QuickPay Settlement API",
  "batches": [
    {
      "batch_id": "B001",
      "merchant": {
        "merchant_id": "M001",
        "merchant_name": "Alpha Mart",
        "region": "APAC"
      },
      "settlements": [
        {
          "settlement_id": "S001",
          "amount_usd": 1520.5,
          "status": "settled",
          "processed_at": "2026-03-07T08:10:00Z",
          "bank": {
            "name": "Bank A",
            "country": "IN"
          }
        },
        {
          "settlement_id": "S002",
          "amount_usd": 980.0,
          "status": "pending",
          "processed_at": "2026-03-07T08:45:00Z",
          "bank": {
            "name": "Bank A",
            "country": "IN"
          }
        },
        {
          "settlement_id": "S003",
          "amount_usd": 640.0,
          "status": "settled",
          "processed_at": "2026-03-07T09:15:00Z",
          "bank": {
            "name": "Bank B",

In [3]:
records = []

for batch in data['batches']:
    batch_id = batch['batch_id']
    merchant_id = batch['merchant']['merchant_id']
    merchant_name = batch['merchant']['merchant_name']
    region = batch['merchant']['region']

    for settlement in batch['settlements']:
        record = {
            'batch_id': batch_id,
            'merchant_id': merchant_id,
            'merchant_name': merchant_name,
            'region': region,
            'settlement_id': settlement['settlement_id'],
            'amount_usd': settlement['amount_usd'],
            'status': settlement['status'],
            'processed_at': settlement['processed_at'],
            'bank_name': settlement['bank']['name'],
            'bank_country': settlement['bank']['country']
        }
        records.append(record)

df = pd.DataFrame(records)

display(df.head())

,batch_id,merchant_id,merchant_name,region,settlement_id,amount_usd,status,processed_at,bank_name,bank_country
0,B001,M001,Alpha Mart,APAC,S001,1520.5,settled,2026-03-07T08:10:00Z,Bank A,IN
1,B001,M001,Alpha Mart,APAC,S002,980.0,pending,2026-03-07T08:45:00Z,Bank A,IN
2,B001,M001,Alpha Mart,APAC,S003,640.0,settled,2026-03-07T09:15:00Z,Bank B,SG
3,B002,M004,Delta Travels,US,S004,2100.0,settled,2026-03-07T08:20:00Z,Bank C,US
4,B002,M004,Delta Travels,US,S005,500.0,failed,2026-03-07T08:50:00Z,Bank C,US


In [4]:
df.columns = df.columns.str.lower().str.replace(' ', '_')

display(df.head())

,batch_id,merchant_id,merchant_name,region,settlement_id,amount_usd,status,processed_at,bank_name,bank_country
0,B001,M001,Alpha Mart,APAC,S001,1520.5,settled,2026-03-07T08:10:00Z,Bank A,IN
1,B001,M001,Alpha Mart,APAC,S002,980.0,pending,2026-03-07T08:45:00Z,Bank A,IN
2,B001,M001,Alpha Mart,APAC,S003,640.0,settled,2026-03-07T09:15:00Z,Bank B,SG
3,B002,M004,Delta Travels,US,S004,2100.0,settled,2026-03-07T08:20:00Z,Bank C,US
4,B002,M004,Delta Travels,US,S005,500.0,failed,2026-03-07T08:50:00Z,Bank C,US


### Convert Data Types

Convert the `processed_at` column to datetime objects to enable time-based analysis.

In [5]:
df['processed_at'] = pd.to_datetime(df['processed_at'])

display(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype              
---  ------         --------------  -----              
 0   batch_id       6 non-null      object             
 1   merchant_id    6 non-null      object             
 2   merchant_name  6 non-null      object             
 3   region         6 non-null      object             
 4   settlement_id  6 non-null      object             
 5   amount_usd     6 non-null      float64            
 6   status         6 non-null      object             
 7   processed_at   6 non-null      datetime64[ns, UTC]
 8   bank_name      6 non-null      object             
 9   bank_country   6 non-null      object             
dtypes: datetime64[ns, UTC](1), float64(1), object(8)
memory usage: 612.0+ bytes


None

### Normalize Numerical Features

Normalize numerical columns, specifically `amount_usd`, using `MinMaxScaler` to scale values between 0 and 1. This is a common preprocessing step for many machine learning algorithms.

In [6]:
from sklearn.preprocessing import MinMaxScaler

# Initialize the MinMaxScaler
scaler = MinMaxScaler()

# Normalize the 'amount_usd' column
df['amount_usd_normalized'] = scaler.fit_transform(df[['amount_usd']])

# Display the DataFrame with the new normalized column
display(df.head())

,batch_id,merchant_id,merchant_name,region,settlement_id,amount_usd,status,processed_at,bank_name,bank_country,amount_usd_normalized
0,B001,M001,Alpha Mart,APAC,S001,1520.5,settled,2026-03-07 08:10:00+00:00,Bank A,IN,0.152313
1,B001,M001,Alpha Mart,APAC,S002,980.0,pending,2026-03-07 08:45:00+00:00,Bank A,IN,0.071642
2,B001,M001,Alpha Mart,APAC,S003,640.0,settled,2026-03-07 09:15:00+00:00,Bank B,SG,0.020896
3,B002,M004,Delta Travels,US,S004,2100.0,settled,2026-03-07 08:20:00+00:00,Bank C,US,0.238806
4,B002,M004,Delta Travels,US,S005,500.0,failed,2026-03-07 08:50:00+00:00,Bank C,US,0.000000


In [8]:
# Save the DataFrame to a CSV file
df.to_csv('normalized_data.csv', index=False)

print("Normalized data saved to 'normalized_data.csv'")

Normalized data saved to 'normalized_data.csv'


In [17]:
ledger_df = pd.read_csv('/content/ledger.csv')
gateway_df = pd.read_csv('/content/gateway.csv')

print("Ledger DataFrame head:")
display(ledger_df.head())

print("\nGateway DataFrame head:")
display(gateway_df.head())

Ledger DataFrame head:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,850.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R004,2026-03-02,M003,2100.0,success,Card
4,R005,2026-03-03,M004,7200.0,success,Card



Gateway DataFrame head:


,transaction_id,transaction_date,merchant_id,amount_usd,status,payment_method
0,R001,2026-03-01,M001,1200.0,success,UPI
1,R002,2026-03-01,M002,900.0,success,Card
2,R003,2026-03-02,M001,500.0,success,Wallet
3,R005,2026-03-03,M004,7200.0,failed,Card
4,R006,2026-03-03,M002,950.0,success,UPI
